In [211]:
from cgra import *
from kernels import *
from sat_to_csv import *

In [212]:
kernel_name = "mmul_os_opt2"
version = ""

In [213]:
# Global variables
CGRA_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr_A = 20000
#first_addr_B = first_addr_A + rowsA*colsA*4
#first_addr_C = first_addr_B + colsA*colsB*4
first_addr_B = 30000
first_addr_C = 40000
end_addr_C = first_addr_C

In [214]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [215]:
# Data
def configMemory(A_data, B_data, rowsA, colsA, colsB):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------
    # &B[0][0]          &C[0][1]        &A[0][0]    nRowsBlocksC
    # nColsBlocksC      &B[0][1]        &C[1][2]    &A[1][0]
    # &A[2][0]          loopColsA       &B[0][2]    &C[2][3]
    # &C[3][0]          &A[3][0]        -           &B[0][3]
    # ----------------------
    # -4*colsB          colsA           -           -
    # -                 -4*colsB        colsA       -
    # -                 -               -4*colsB    colsA
    # colsA             -               -           -4*colsB
    nItLoopColsA = colsA
    nColsBlocksC = int(colsB/CGRA_N_ROWS)
    nRowsBlocksC = int(rowsA/CGRA_N_ROWS)
    config_vals_col0 = [first_addr_B, nColsBlocksC, first_addr_A + 2*colsA*4, first_addr_C + 3*colsB*4, -4*colsB, colsA]
    config_vals_col1 = [first_addr_C + 4, first_addr_B + 4, nItLoopColsA, first_addr_A + 3*colsA*4, colsA, -4*colsB]
    config_vals_col2 = [first_addr_A, first_addr_C + 2*4 + colsB*4, first_addr_B + 2*4, colsA, -4*colsB]
    config_vals_col3 = [nRowsBlocksC, first_addr_A + colsA*4, first_addr_C + 3*4 + 2*colsB*4, first_addr_B + 3*4, colsA, -4*colsB]
    addr_config_loads_col0 = 0
    kernel_add_memory_region(kernel_name, addr_config_loads_col0, config_vals_col0, version=version)
    addr_config_loads_col1 = addr_config_loads_col0 + len(config_vals_col0)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col1, config_vals_col1, version=version)
    addr_config_loads_col2 = addr_config_loads_col1 + len(config_vals_col1)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col2, config_vals_col2, version=version)
    addr_config_loads_col3 = addr_config_loads_col2 + len(config_vals_col2)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col3, config_vals_col3, version=version)
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_B, B_data, version=version)
    # Config data address for direct loads
    load_addrs = [addr_config_loads_col0, addr_config_loads_col1, addr_config_loads_col2, addr_config_loads_col3]
    return load_addrs

In [216]:
def runKernel(load_addrs, max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [217]:
def getResult(first_addr_C, end_addr_C):
    result = []
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result.append(int(row[1]))
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [218]:
def mmul_cpu(A_data, B_data, rowsA, colsA, colsB):
    expected_res = [0 for _ in range(rowsA*colsB)]
    for rA in range(rowsA):
        for cB in range(colsB):
            sum = 0
            for cA in range(colsA):
                sum += A_data[rA*colsA + cA] * B_data[cA*colsB + cB]
            expected_res[rA*colsB + cB] = sum
    return expected_res

In [219]:
# Test dimensions (4xXx4)
rowsA = 16
colsA = 7
colsB = 32
A_data = list(range(0, rowsA * colsA))
B_data = [x + 100 for x in range(0, colsA * colsB)]
print("A")
printAsMatrix(A_data, rowsA, colsA)
print("B")
printAsMatrix(B_data, colsA, colsB)
load_addrs = configMemory(A_data, B_data, rowsA, colsA, colsB)

A
[0, 1, 2, 3, 4, 5, 6]
[7, 8, 9, 10, 11, 12, 13]
[14, 15, 16, 17, 18, 19, 20]
[21, 22, 23, 24, 25, 26, 27]
[28, 29, 30, 31, 32, 33, 34]
[35, 36, 37, 38, 39, 40, 41]
[42, 43, 44, 45, 46, 47, 48]
[49, 50, 51, 52, 53, 54, 55]
[56, 57, 58, 59, 60, 61, 62]
[63, 64, 65, 66, 67, 68, 69]
[70, 71, 72, 73, 74, 75, 76]
[77, 78, 79, 80, 81, 82, 83]
[84, 85, 86, 87, 88, 89, 90]
[91, 92, 93, 94, 95, 96, 97]
[98, 99, 100, 101, 102, 103, 104]
[105, 106, 107, 108, 109, 110, 111]
B
[100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131]
[132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163]
[164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195]
[196, 197, 198, 199, 200, 201, 202, 203, 204, 2

In [220]:
runKernel(load_addrs, max_it=1000*rowsA)

Instr =  0 ( 0 )
[30000, 40004, 20000,    4]    [LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4]    
[   8, 30004, 40136, 20028]    [LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4]    
[20056,    7, 30008, 40268]    [LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4]    
[40384, 20084,    0, 30012]    [LWD R2  4, LWD R2  4, NOP , LWD R2  4]    
-------
Instr =  1 ( 1 )
[-128,    7, 20000,    0]    [LWD R1  4, LWD R1  4, NOP , SADD R1  ZERO  ZERO]    
[   0, -128,    7, 20028]    [SADD R1  ZERO  ZERO, LWD R1  4, LWD R1  4, NOP ]    
[20056,    0, -128,    7]    [NOP , SADD R1  ZERO  ZERO, LWD R1  4, LWD R1  4]    
[   7, 20084,    0, -128]    [LWD R1  4, NOP , NOP , LWD R1  4]    
-------
Instr =  2 ( 2 )
[   0,    0,    0,    0]    [SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO]    
[   0,    0,    0,    0]    [SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO]    
[   0,    0,    0,    0]    [SADD R3  ZERO  ZERO, SADD R3  ZERO  ZE

In [221]:
result = getResult(first_addr_C, first_addr_C + rowsA*colsB*4)
expected_res = mmul_cpu(A_data, B_data, rowsA, colsA, colsB)

errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
print("Err: " + str(errors))
if errors > 0:
    print("CGRA res: " + str(len(result)))
    printAsMatrix(result, rowsA, colsB)
    print("CPU res: " + str(len(expected_res)))
    printAsMatrix(expected_res, rowsA, colsB)


Err: 0
